# Data Understanding

In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('China Automobile Sales Data.csv')
df.head()

,model,units_sold,make,low_price,high_price,year_month,is_ev,body_type,brand,brand_country
0,Model Y,69098,特斯拉中国,316.9,417.9,2022-11-01,EV,SUV,Tesla,United States
1,宏光MINIEV,68567,上汽通用五菱,32.8,99.9,2022-11-01,EV,Hatchback,Wuling Motors,China
2,宋PLUS新能源,64145,比亚迪,152.8,216.8,2022-11-01,EV,SUV,BYD,China
3,汉,31786,比亚迪,214.8,329.8,2022-11-01,EV,Sedan,BYD,China
4,Model 3,31193,特斯拉中国,266.7,339.9,2022-11-01,EV,Sedan,Tesla,United States


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38806 entries, 0 to 38805
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   model          38806 non-null  object 
 1   units_sold     38806 non-null  int64  
 2   make           38806 non-null  object 
 3   low_price      38806 non-null  float64
 4   high_price     38806 non-null  float64
 5   year_month     38806 non-null  object 
 6   is_ev          38806 non-null  object 
 7   body_type      36440 non-null  object 
 8   brand          38806 non-null  object 
 9   brand_country  38806 non-null  object 
dtypes: float64(2), int64(1), object(7)
memory usage: 3.0+ MB


In [9]:
df.shape

(38806, 10)

## Missing values and duplicates

In [13]:
df.isnull().sum()

model               0
units_sold          0
make                0
low_price           0
high_price          0
year_month          0
is_ev               0
body_type        2366
brand               0
brand_country       0
dtype: int64

In [14]:
df.duplicated().sum()

np.int64(33)

## Summary statistics

In [5]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
model,38806,1321,风光S560,77,NaN,NaN,NaN,NaN,NaN,NaN,NaN
units_sold,38806.0,NaN,NaN,NaN,3527.105422,5882.556188,1.0,258.0,1217.0,4103.0,73009.0
make,38806,175,长安汽车,1688,NaN,NaN,NaN,NaN,NaN,NaN,NaN
low_price,38806.0,NaN,NaN,NaN,139.083858,115.803772,0.0,65.9,119.8,182.7,1365.8
high_price,38806.0,NaN,NaN,NaN,185.846993,151.873792,0.0,92.9,159.8,256.8,1686.0
year_month,38806,76,2023-12-01,566,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_ev,38806,2,Gasoline,30278,NaN,NaN,NaN,NaN,NaN,NaN,NaN
body_type,36440,5,SUV,18762,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand,38806,139,Volkswagen,2192,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand_country,38806,10,China,21544,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Outlier scan (numeric columns)

In [7]:
import numpy as np
from pandas.api.types import is_numeric_dtype

numeric_cols = [c for c in df.columns if is_numeric_dtype(df[c])]
outlier_rows = []
for col in numeric_cols:
    series = df[col].dropna()
    if series.empty:
        continue
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_rows.append({
        'column': col,
        'outlier_count': int(outliers),
        'outlier_pct': round(outliers / len(df) * 100, 2)
    })
pd.DataFrame(outlier_rows).sort_values('outlier_count', ascending=False).head(20)

,column,outlier_count,outlier_pct
0,units_sold,4096,10.56
1,low_price,2005,5.17
2,high_price,1456,3.75


## Quick insights (auto-generated)

In [8]:
from pandas.api.types import is_numeric_dtype, is_datetime64_any_dtype

numeric_cols = [c for c in df.columns if is_numeric_dtype(df[c])]
object_cols = [c for c in df.columns if df[c].dtype == 'object']

candidate_date_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in ['date', 'month', 'year', 'time'])
]
date_col = None
date_col_is_year = False
for col in candidate_date_cols:
    if is_datetime64_any_dtype(df[col]):
        date_col = col
        break
    if is_numeric_dtype(df[col]) and 'year' in col.lower():
        year_series = df[col].dropna()
        if not year_series.empty and (year_series.between(1900, 2100).mean() >= 0.8):
            date_col = col
            date_col_is_year = True
            break
    parsed = pd.to_datetime(df[col], errors='coerce', infer_datetime_format=True)
    if parsed.notna().mean() >= 0.8:
        date_col = col
        break

insights = []
insights.append(f'Rows: {len(df):,}; Columns: {df.shape[1]:,}')

if numeric_cols:
    stds = df[numeric_cols].std(numeric_only=True).sort_values(ascending=False)
    insights.append('Most variable numeric columns: ' + ', '.join(stds.head(3).index))

missing_pct = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)
top_missing = missing_pct[missing_pct > 0].head(3)
if not top_missing.empty:
    insights.append('Highest missingness: ' + '; '.join([
        f'{idx} ({val:.1f}%)' for idx, val in top_missing.items()
    ]))

if object_cols:
    top_col = object_cols[0]
    top_values = df[top_col].value_counts(dropna=False).head(3)
    insights.append('Top categories in ' + top_col + ': ' + ', '.join([
        f'{idx} ({val})' for idx, val in top_values.items()
    ]))

if date_col and numeric_cols:
    temp = df[[date_col, numeric_cols[0]]].copy()
    if date_col_is_year:
        temp[date_col] = pd.to_datetime(temp[date_col].astype('Int64').astype(str), format='%Y', errors='coerce')
    elif not is_datetime64_any_dtype(temp[date_col]):
        temp[date_col] = pd.to_datetime(temp[date_col], errors='coerce', infer_datetime_format=True)
    temp = temp.dropna(subset=[date_col])
    if not temp.empty:
        temp = temp.set_index(date_col).sort_index()
        series = temp[numeric_cols[0]].resample('M').mean()
        if not series.empty:
            insights.append(f'{numeric_cols[0]} date range: {series.index.min().date()} to {series.index.max().date()}')
            insights.append(f'{numeric_cols[0]} latest 3-month avg: {series.tail(3).mean():.2f}')

for i, insight in enumerate(insights, start=1):
    print(f'{i}. {insight}')

1. Rows: 38,806; Columns: 10
2. Most variable numeric columns: units_sold, high_price, low_price
3. Highest missingness: body_type (6.1%)
4. Top categories in model: 风光S560 (77), 起亚K3 (76), 骐达TIIDA (76)
5. units_sold date range: 2018-01-31 to 2024-04-30
6. units_sold latest 3-month avg: 2628.20


C:\Users\PHUC\AppData\Local\Temp\ipykernel_6476\3206665253.py:22: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(df[col], errors='coerce', infer_datetime_format=True)
C:\Users\PHUC\AppData\Local\Temp\ipykernel_6476\3206665253.py:53: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  temp[date_col] = pd.to_datetime(temp[date_col], errors='coerce', infer_datetime_format=True)
C:\Users\PHUC\AppData\Local\Temp\ipykernel_6476\3206665253.py:57: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  